In [1]:
import numpy as np

In [ ]:
np.random.seed(42)

# ─── Architecture ───────────────────────────────────────────────────
# Conv1 : 8 filters, 2 in-ch, 3x3  → 8*2*9 = 144 weights
# Conv2 : 16 filters, 8 in-ch, 3x3 → 16*8*9 = 1152 weights
# FC1   : 32 neurons, 256 inputs    → 32*256 = 8192 weights
# FC2   : 4 neurons,  32 inputs     → 4*32   = 128 weights
#
# Weight scale: mean ~28, range [10,50]
# With threshold=128, beta=200:
#   avg active inputs in 3x3 window ~4 (15% density input)
#   4 * 28 = 112  <  128  → fires after ~2 timesteps of accumulation ✓
#   For FC1: 256 inputs, ~15% active = 38 inputs, 38*28=1064 >> 128
#   → scale FC weights down: FC1 mean=4, FC2 mean=40

def write_hex(arr, filename, bits=16):
    """Write flat array as hex words, one per line (for $readmemh)"""
    mask = (1 << bits) - 1
    with open(filename, 'w') as f:
        for val in arr.flatten():
            f.write(f"{int(val) & mask:04x}\n")
    print(f"  {filename}: {arr.size} words")

# Conv1 weights  [out_ch, in_ch, kH, kW]
w_conv1 = np.random.randint(10, 50, size=(8, 2, 3, 3)).astype(np.int32)
# Conv2 weights  [out_ch, in_ch, kH, kW]
w_conv2 = np.random.randint(10, 50, size=(16, 8, 3, 3)).astype(np.int32)
# FC1 weights    [out, in]  — smaller to prevent saturation
w_fc1   = np.random.randint(2, 8,  size=(32, 256)).astype(np.int32)
# FC2 weights    [out, in]
w_fc2   = np.random.randint(25, 55, size=(4, 32)).astype(np.int32)

print("Writing weight hex files...")
write_hex(w_conv1, "/home/claude/weights_conv1.hex")
write_hex(w_conv2, "/home/claude/weights_conv2.hex")
write_hex(w_fc1,   "/home/claude/weights_fc1.hex")
write_hex(w_fc2,   "/home/claude/weights_fc2.hex")

# ─── Verify sizes match what RTL expects ────────────────────────────
print("\nExpected BRAM word counts:")
print(f"  Conv1 weight bus = {8*2*9*16} bits = {8*2*9} words of 16b")
print(f"  Conv2 weight bus = {16*8*9*16} bits = {16*8*9} words of 16b")
print(f"  FC1  BRAM words  = {32*256}  (32 neurons × 256 inputs)")
print(f"  FC2  BRAM words  = {4*32}   (4 neurons × 32 inputs)")

# ─── Print a few stats ──────────────────────────────────────────────
print(f"\nWeight stats:")
print(f"  conv1 : min={w_conv1.min()} max={w_conv1.max()} mean={w_conv1.mean():.1f}")
print(f"  conv2 : min={w_conv2.min()} max={w_conv2.max()} mean={w_conv2.mean():.1f}")
print(f"  fc1   : min={w_fc1.min()}  max={w_fc1.max()}  mean={w_fc1.mean():.1f}")
print(f"  fc2   : min={w_fc2.min()} max={w_fc2.max()} mean={w_fc2.mean():.1f}")

print("\nDone.")

In [3]:
import numpy as np

np.random.seed(42)

CONV1_FILT, IN_CH = 8, 2
CONV2_FILT = 16
FC1_OUT, POOL2_OUT = 32, 256
FC2_OUT = 4
SCALE = 256

def write_q8_8_hex(arr, filename):
    scaled = np.round(arr * SCALE).astype(np.int32)
    mask = 0xFFFF
    with open(filename, "w") as f:
        for val in scaled.flatten():
            f.write(f"{int(val) & mask:04x}\n")

w_conv1 = np.random.uniform(0.1, 4.0, size=(CONV1_FILT, IN_CH, 3, 3))
w_conv2 = np.random.uniform(0.05, 2.0, size=(CONV2_FILT, CONV1_FILT, 3, 3))
w_fc1 = np.random.uniform(0.01, 0.6, size=(FC1_OUT, POOL2_OUT))
w_fc2 = np.random.uniform(0.1, 3.0, size=(FC2_OUT, FC1_OUT))

write_q8_8_hex(w_conv1, "weights_conv1.hex")
write_q8_8_hex(w_conv2, "weights_conv2.hex")
write_q8_8_hex(w_fc1, "weights_fc1.hex")
write_q8_8_hex(w_fc2, "weights_fc2.hex")